# Subclassing numpy.memmap

To circumvent the restriction on `memmap` from the max number of open file, it may be possible to subclass either the `memmap` or the `ndarray` class, so it has a filename, offset, and shape attribute, which are then used when one tries to actually make use of the data.

The relevant reference here is [Subclassing ndarray](https://docs.scipy.org/doc/numpy-1.13.0/user/basics.subclassing.html).

As per the tests below, the definition call to `numpy.memmap`calls `__new__`, while when using the resulting array in an expression, the `__array_prepare__` procedure is called.   This may be used to close the `_mmap` in `__new__`, while saving sufficient data in a new attribute to make it possible to reopen the `_mmap` object in the call to `_array_prepare__`

In [1]:
import numpy as np
import mmap

class memmap2(np.memmap):

    def __new__(subtype, *args, verbose=False, **kwargs):
        obj = super(memmap2, subtype).__new__(subtype,  *args, **kwargs)
        obj._saved = {'len':obj._mmap.__len__(),'off':obj.offset,
                      'file':obj.filename,'verbose':verbose}
        fo=open(obj.filename,'rb+')
        obj._mmap.close()
        return obj
    
    def reopen(self):
        print('reopen')
        s=self._saved
        if self._mmap.closed:
            print('closed')
            if self._saved['verbose']:
                print('was closed, reopen')
            fo=open(s['file'],'rb+')
            block=mmap.ALLOCATIONGRANULARITY
            o=(s['off']//block)*block
            self._mmap=mmap.mmap(fo.fileno(),length=s['len'],offset=o)
        else:
            print('open')
            if self._saved['verbose']:
                print('was open')

    def __subclasshook__(self, *arg, **kwargs):
        self.reopen()
        obj = super(memmap2, self).__subclasshook__(*arg, **kwargs)
        return obj

    def __array_finalize__ (self, obj):
        super(memmap2,self).__array_finalize__(obj)
        if obj is None:
            return
        self._saved = obj._saved
        self._mmap = obj._mmap

    def __str__(self):
        if self._verbose:
            print('-> memmap3.__str__')
        self.reopen()
        obj = super(memmap3,self).__str__()
        return obj


In [2]:
import numpy as np
import mmap
class memmap3(np.memmap):

    def __new__(subtype, file, dtype=np.float32, mode='w+', offset=0, 
                shape=(10,), order='F',verbose=False,close=False):
        self = super(memmap3, subtype).__new__(subtype, file, dtype=dtype, 
                    mode=mode, offset=offset, shape=shape, order=order)
        self._verbose=verbose
        self._close=close
        self._saved=(file,self._mmap.__len__(),self.offset)
        self._mmap.close()
        if self._verbose:
            print('-> memmap3.__new__: saved =',self._saved,type(self.offset),type(offset))
        return self

    def reopen(self):
        (file,length,offset)=self._saved
        if self._verbose:
            print("-> memmap3.reopen: length, offset =",length,offset)
        if self._mmap.closed:
            if self._verbose:
                print('--> was closed, reopen')
            fo=open(file,'rb+')
            self._fileobj=fo
            block=mmap.ALLOCATIONGRANULARITY
            o=(offset//block)*block
            self._mmap=mmap.mmap(fo.fileno(),length,offset=o)
        else:
            if self._verbose:
                print('--> was already open')

    def close(self):
        if self._close:
            self._mmap.close()
            self._fileobj.close()

    def __array_wrap__(self, arr, context=None):
        if self._verbose:
            print('-> memmap3.__array_wrap__')
        arr = super(memmap3, self).__array_wrap__(arr, context)
        return arr

    def __array_prepare__ (self, obj, context=None):
        if self._verbose:
            print('-> memmap3.__array_prepare__')
        self.reopen()
        obj = super(memmap3, self).__array_prepare__(obj, context)
        self.close()
        return obj

    def __array_finalize__ (self, obj):
        super(memmap3,self).__array_finalize__(obj)
        if obj is None:
            return
        self._saved = obj._saved
        self._mmap = obj._mmap

    def __repr__(self):
        if self._verbose:
            print('-> memmap3.__repr__')
        self.reopen()
        output = super(memmap3,self).__repr__()
        self.close()
        return output

    def __str__(self):
        if self._verbose:
            print('-> memmap3.__str__')
        self.reopen()
        output = super(memmap3,self).__str__()
        self.close()
        return output


In [3]:
import numpy as np
import mmap
class memmap4(np.memmap):
    def __new__(subtype, file, dtype=np.float32,mode='w+',offset=0, 
                shape=(10,), order='F',verbose=False,close=False):
        self = super(memmap4, subtype).__new__(subtype,file,dtype=dtype, 
                    mode=mode,offset=offset,shape=shape,order=order)
        self._verbose=verbose
        self._close=close
        self._saved=(file,self._mmap.__len__(),self.offset)
        self._mmap.close()
        if self._verbose:
            print('-> memmap4.__new__: saved =',self._saved,
                  type(self.offset),type(offset))
        return self
    def verbose(self):
        return hasattr(self,'_verbose') and self._verbose
    def reopen(self):
        (file,length,offset)=self._saved
        if self.verbose():
            print("-> memmap4.reopen: length, offset =",length,offset)
        if self._mmap.closed:
            if self.verbose():
                print('--> was closed, reopen')
            fo=open(file,'rb+')
            self._fileobj=fo
            block=mmap.ALLOCATIONGRANULARITY
            o=(offset//block)*block
            self._mmap=mmap.mmap(fo.fileno(),length,offset=o)
        else:
            if self.verbose():
                print('--> was already open')
    def __array_wrap__(self, arr, context=None):
        if self.verbose():
            print('-> memmap4.__array_wrap__')
        arr = super(memmap4, self).__array_wrap__(arr, context)
        return arr
    def __array_prepare__ (self, obj, context=None):
        if self.verbose():
            print('-> memmap4.__array_prepare__')
        self.reopen()
        obj = super(memmap4, self).__array_prepare__(obj, context)
        return obj
    def __array_finalize__ (self, obj):
        super(memmap4,self).__array_finalize__(obj)
        if self.verbose():
            print('-> memmap4.__array_finalize__')
        if obj is None:
            return
        self._saved = obj._saved
        self._mmap = obj._mmap
    def __repr__(self):
        self.reopen()
        if self.verbose():
            print('-> memmap4.__repr__')
        output = super(memmap4,self).__repr__()
        return output
    def __str__(self):
        if self.verbose():
            print('-> memmap4.__str__')
        self.reopen()
        output = super(memmap4,self).__str__()
        return output

Open a `numpy.memmap` for writing, write some values, and show that the file is still open:

In [4]:
pwd

'/lustre/hpc/astro/aake/codes/DISPATCH2_clean/tests/python/memmap'

In [5]:
s=(10000,)
mm=np.memmap('tmp.dat',mode='w+',order='F',dtype=np.float32,shape=s)
for i in range(s[0]):
    mm[i]=i
(mm,mm._mmap.closed)

(memmap([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
         9.999e+03], dtype=float32), False)

Open for reading, and show the same:

In [6]:
mm=np.memmap('tmp.dat',mode='r+',order='F',dtype=np.float32,shape=(3,),
             offset=4*4000)
(mm,mm._mmap.closed)

(memmap([4000., 4001., 4002.], dtype=float32), False)

## class memmap4 tests

Open the `memmap4` subclass for reading, and show that the file is closed

In [7]:
mm2=memmap4('tmp.dat',mode='r+',order='F',dtype=np.float32,shape=(10,),
           verbose=True)
(mm2._saved,mm2._mmap.closed)

-> memmap4.__new__: saved = ('tmp.dat', 40, 0) <class 'int'> <class 'int'>


(('tmp.dat', 40, 0), True)

Show that `print` and "last value" use different object methods:

In [8]:
print(mm2)
mm2

-> memmap4.__str__
-> memmap4.reopen: length, offset = 40 0
--> was closed, reopen
[0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]
-> memmap4.reopen: length, offset = 40 0
--> was already open
-> memmap4.__repr__


memmap4([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.], dtype=float32)

## class memmap3 tests

Open the `memmap2` subclass for reading, and show that the file is closed.  The reference to the variable `mm3` will result in the file being opened, and the (default) choice `close=False` will result in the file being left open

In [10]:
mm3=memmap3('tmp.dat',mode='r+',order='F',dtype=np.float32,shape=(3,),
            offset=12,verbose=0,close=False)
mm3._saved,mm3._mmap.closed,mm3

(('tmp.dat', 24, 12), True, memmap3([3., 4., 5.], dtype=float32))

In [11]:
(mm3._mmap.closed,mm3)

(False, memmap3([3., 4., 5.], dtype=float32))

Choosing instead `close=True`, or setting the attribute `_close=True`, will result in both the mmap and the file being closed, and then being kept closed:

In [12]:
mm3._close=True

In [13]:
for i in range(4):
    print(mm3._mmap.closed,mm3)

False [3. 4. 5.]
True [3. 4. 5.]
True [3. 4. 5.]
True [3. 4. 5.]


## Cost comparisons

Compute the cost of `np.memmap`:

In [16]:
import time
start=time.time()
mms=[]
nj=1
ni=900
m=0
for j in range(nj):
    for i in range(1,ni+1):
        o=4*i
        mm=np.memmap('tmp.dat',mode='r+',order='F',dtype=np.float32,
                    shape=(6,),offset=o)
        mms.append(mm)
        m+=1
        if m%30000==0:
            print(m,j,i,mm)
print('{} memmaps. One memmap takes about {:.3f} ms'.format(
    len(mms),(time.time()-start)*1e3/(ni*nj)))

900 memmaps. One memmap takes about 0.176 ms


Compute the cost of `memmap4`:

In [18]:
import time
start=time.time()
mms=[]
nj=10
ni=9000
m=0
for j in range(nj):
    for i in range(1,ni+1):
        o=4*i
        mm=memmap4('tmp.dat',mode='r+',order='F',dtype=np.float32,
                    shape=(6,),offset=o,verbose=0,close=False)
        mms.append(mm)
        m+=1
        if m%30000==0:
            print(m,j,i,mm)
print('one memmap4 takes about {:.3f} ms'.format(
    (time.time()-start)*1e3/(ni*nj)))

30000 3 3000 [3000. 3001. 3002. 3003. 3004. 3005.]
60000 6 6000 [6000. 6001. 6002. 6003. 6004. 6005.]
90000 9 9000 [9000. 9001. 9002. 9003. 9004. 9005.]
one memmap4 takes about 0.117 ms


### Clean up, removing the files

In [19]:
import os
mm._mmap.close()
mm._fileobj.close()
os.remove('tmp.dat')

List the files in the current directory we have open

In [23]:
import os
import psutil
cwd=os.getcwd()
fopen=psutil.Process().open_files()
fopen

[popenfile(path='/lustre/hpc/astro/aake/codes/jupyter.log', fd=1, position=17405, mode='w', flags=32769),
 popenfile(path='/lustre/hpc/astro/aake/codes/jupyter.log', fd=2, position=17405, mode='w', flags=32769),
 popenfile(path='/lustre/hpc/astro/aake/.ipython/profile_default/history.sqlite', fd=36, position=1445888, mode='r+', flags=557058),
 popenfile(path='/lustre/hpc/astro/aake/.ipython/profile_default/history.sqlite', fd=38, position=40, mode='r+', flags=557058),
 popenfile(path='/lustre/hpc/astro/aake/.ipython/profile_default/history.sqlite-journal', fd=52, position=0, mode='r+', flags=557058)]

In [24]:
for f,fd,position,mode,flags in fopen:
    if os.path.dirname(f)==cwd:
        print(os.path.basename(f))  

### Remove all `tmp*` files

In [25]:
print(os.getcwd())

/lustre/hpc/astro/aake/codes/DISPATCH2_clean/tests/python/memmap


In [26]:
files=[f for f in os.listdir() if f.startswith('tmp')]
files

[]

In [27]:
for f in files:
    os.remove(f)